# Train 3DQN + Max Pressure (Hybrid) trên GAMA — Google Colab Pro

Notebook chạy `dqn_py/train.py --algorithm hybrid --site lvl` (nút Lê Văn Lương × Hoàng Đạo Thúy) với nhiều GAMA headless worker (CPU) và một learner trên GPU.

**Cách dùng**
1. *Runtime → Change runtime type*: **T4 GPU** + bật **High-RAM**.
2. Sửa ô cấu hình nếu cần, rồi *Runtime → Run all*.
3. Bị ngắt / hết giờ: mở lại notebook, **Run all** lần nữa — tự resume từ checkpoint trên Drive.

**Lưu ý**
- Colab Pro không chạy nền: giữ tab mở.
- Lần đầu train, cấu hình được ghi vào `run_spec.json` trên Drive; các lần resume dùng lại đúng file đó (cùng commit, cùng số worker), vì `train.py` từ chối resume khi cấu hình hoặc code khác. Muốn đổi cấu hình → đổi `RUN_NAME`.
- Kết quả train nằm ở local disk `/content/runs/` và được chép lên Drive mỗi `SYNC_MINUTES` phút; mất kết nối thì chỉ mất phần sau lần chép cuối.

In [ ]:
# ===== Cấu hình (chỉ có hiệu lực ở lần train đầu tiên của RUN_NAME) =====
SITE = "lvl"                  # lvl: Lê Văn Lương × Hoàng Đạo Thúy | doan: mạng GAMA cũ (Nguyễn Trãi – KĐT)
REWARD = "r1"                 # r1 | r2 | r3 | r4 (xem README)
RUN_NAME = f"hybrid_{SITE}_{REWARD}"
PRESSURE_LAMBDA = 0.5         # sức ảnh hưởng của điểm Max Pressure
TOTAL_STEPS = 1_000_000       # số quyết định của agent
MAX_HOURS = 20                # mỗi session dừng sạch sau 20h, trước giới hạn 24h của Colab Pro
WORKERS = None                # None = số CPU - 1
EXTRA_ARGS = []               # tham số train.py thêm, vd ["--validate-every", "100"]
SYNC_MINUTES = 15
RUN_SMOKE_TEST = True         # chạy thử 1 episode ngắn để chắc GAMA chạy được trên runtime này
REPO_URL = "https://github.com/Thanh-2312/ColabTraffict.git"  # fork chứa model LVL; push code mới lên đây trước
GAMA_URL = "https://github.com/gama-platform/gama/releases/download/2025.06.4/GAMA_2025.06.4_Linux_with_JDK.zip"
DRIVE_ROOT = "/content/drive/MyDrive/Traffic-Light-ABM/runs"

In [ ]:
import glob, json, os, shlex, signal, subprocess
from google.colab import drive

def run(cmd):
    # Lệnh ngắn: in output, dừng notebook nếu lỗi.
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-3000:], r.stderr[-3000:], sep="", end="")
    if r.returncode:
        raise RuntimeError(f"Lỗi ({r.returncode}): {cmd}")
    return r.stdout

def stream(cmd):
    # Lệnh dài: output hiện trực tiếp trong ô.
    get_ipython().system(f"{cmd}; echo $? > /tmp/rc")
    rc = int(open("/tmp/rc").read())
    if rc:
        raise RuntimeError(f"Lỗi ({rc}): {cmd}")

drive.mount("/content/drive")
DRIVE_RUN = f"{DRIVE_ROOT}/{RUN_NAME}"
LOCAL_RUN = f"/content/runs/{RUN_NAME}"
SPEC_FILE = f"{DRIVE_RUN}/run_spec.json"
REPO = "/content/Traffic-Light-ABM"
os.makedirs(DRIVE_RUN, exist_ok=True)

cpus = os.cpu_count()
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'Không có GPU: learner chạy CPU'")
print(f"CPU: {cpus} | RAM: {ram_gb:.0f} GB")

if os.path.exists(SPEC_FILE):
    spec = json.load(open(SPEC_FILE))
    print("Resume: dùng cấu hình đã lưu, bỏ qua ô cấu hình phía trên.")
else:
    workers = WORKERS or max(1, cpus - 1)
    spec = {"commit": None, "workers": workers, "train_args": [
        "--algorithm", "hybrid", "--site", SITE, "--reward-mode", REWARD,
        "--pressure-lambda", str(PRESSURE_LAMBDA),
        "--workers", str(workers), "--start-server", "--port", "6930",
        "--total-steps", str(TOTAL_STEPS), "--episode-horizon", "8000",
        "--max-train-seconds", str(MAX_HOURS * 3600), *EXTRA_ARGS]}
if spec["workers"] + 1 > cpus:
    print(f"⚠ {spec['workers']} worker + 1 learner > {cpus} CPU: vẫn chạy nhưng chậm.")
if spec["workers"] * 2.5 + 4 > ram_gb:
    print(f"⚠ Mỗi worker cần ~2,5 GB RAM: bật High-RAM hoặc giảm WORKERS (đổi RUN_NAME).")
print(json.dumps(spec, indent=1))

In [ ]:
# Code dự án: lần đầu lấy commit mới nhất, các lần resume checkout đúng commit đã lưu.
if not os.path.isdir(REPO):
    run(f"git clone -q {REPO_URL} {REPO}")
if spec["commit"]:
    run(f"git -C {REPO} fetch -q origin && git -C {REPO} checkout -q {spec['commit']}")
else:
    spec["commit"] = run(f"git -C {REPO} rev-parse HEAD").strip()
# Colab đã có torch/numpy/pandas/matplotlib. Không cài requirements.txt: nó hạ numpy xuống 1.x.
# gama-client kéo websockets 10.4, pip sẽ in "dependency conflicts" cho google-genai, langsmith...: bỏ qua được.
run("pip install -q gama-client")
run("which rsync >/dev/null || (apt-get -qq update && apt-get -qq install -y rsync)")

In [ ]:
# GAMA headless cho Linux (kèm JDK).
if not glob.glob("/opt/gama/**/gama-headless.sh", recursive=True):
    run(f"wget -q -O /tmp/gama.zip {GAMA_URL} && unzip -q -o /tmp/gama.zip -d /opt/gama && rm /tmp/gama.zip")
    run("find /opt/gama -path '*/bin/*' -type f -exec chmod +x {} +")
os.environ["GAMA_HEADLESS"] = glob.glob("/opt/gama/**/gama-headless.sh", recursive=True)[0]
print("GAMA_HEADLESS =", os.environ["GAMA_HEADLESS"])

In [ ]:
# Chạy thử: 1 GAMA worker, 1 episode 100 s mô phỏng. Lỗi ở đây thì xem log GAMA in ra bên dưới.
if RUN_SMOKE_TEST:
    try:
        stream(f"rm -rf /content/smoke && python {REPO}/dqn_py/train.py --algorithm hybrid --site {spec['train_args'][3]} "
               f"--reward-mode {REWARD} --total-steps 10 --episodes 1 --episode-horizon 1000 "
               "--warmup 2 --batch-size 2 --buffer-size 32 --start-server --port 6900 "
               "--output-dir /content/smoke")
    except RuntimeError:
        run("tail -n 40 /content/smoke/server_*.log || true")
        raise
    print("✅ Smoke test OK")

In [ ]:
# Train chính. Kéo checkpoint từ Drive về (nếu có), chép kết quả lên Drive định kỳ và khi kết thúc.
if os.path.exists(f"{DRIVE_RUN}/training_state.pt"):
    run(f"mkdir -p {LOCAL_RUN} && rsync -a -u --exclude '.*' {DRIVE_RUN}/ {LOCAL_RUN}/")
if os.path.exists(f"{LOCAL_RUN}/training_state.pt"):
    mode = ["--resume-dir", LOCAL_RUN]
else:
    run(f"rm -rf {LOCAL_RUN}")  # chưa có checkpoint nào thì train.py chỉ chạy được từ đầu
    mode = ["--output-dir", LOCAL_RUN]
json.dump(spec, open(SPEC_FILE, "w"), indent=1)

# training_state.pt chép trước để training_log.csv trên Drive không bao giờ cũ hơn nó (resume cần vậy).
sync = (f"mkdir -p {LOCAL_RUN}; rsync -a {LOCAL_RUN}/training_state.pt {DRIVE_RUN}/ 2>/dev/null; "
        f"rsync -a --exclude server_ws --exclude '*.tmp' {LOCAL_RUN}/ {DRIVE_RUN}/")
syncer = subprocess.Popen(["bash", "-c", f"while sleep {SYNC_MINUTES * 60}; do {sync}; done"],
                          start_new_session=True)
try:
    stream("python " + shlex.join([f"{REPO}/dqn_py/train.py", *spec["train_args"], *mode]))
finally:
    os.killpg(syncer.pid, signal.SIGTERM)
    run(sync)
    print("Đã chép kết quả lên", DRIVE_RUN)

## Kết quả

Trong `DRIVE_ROOT/RUN_NAME` trên Drive:
- `best_weights.pt`: policy tốt nhất theo 2 seed validation (chỉ có sau lượt validation đầu tiên).
- `dqn_weights.pt`: policy cuối; `training_state.pt`: replay + optimizer để resume.
- `training_log.csv`, `validation_log.csv`, `server_*.log`.

Đánh giá trên seed test (chạy trên Colab hoặc máy có GAMA):
```bash
python dqn_py/evaluate.py --site lvl --weights <đường dẫn>/best_weights.pt --episodes 5 --start-server
```